# 01 · Data preparation

**Track 2 – Popularity prediction**

This notebook takes the Kaggle *Spotify Tracks* file as downloaded and turns it into
`data/processed/spotify_clean.csv`, the table the modeling notebooks start from.
The cleaning functions themselves live in `src/data_prep.py`; this notebook walks through
what we found in the raw data and why each step is there.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data_prep import (
    AUDIO_FEATURES, load_raw, prepare_dataset, save_outputs,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. First look at the raw file

In [2]:
raw = load_raw()
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

114,000 rows x 20 columns


,song_id,artists,album_name,song_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,song_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.032,0.000,0.358,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.076,0.924,0.000,0.101,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.056,0.210,0.000,0.117,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.060,0,-18.515,1,0.036,0.905,0.000,0.132,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.053,0.469,0.000,0.083,0.167,119.949,4,acoustic


In [3]:
raw.dtypes.to_frame("dtype").T

,song_id,artists,album_name,song_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,song_genre
dtype,object,object,object,object,int64,int64,bool,float64,float64,int64,float64,int64,float64,float64,float64,float64,float64,float64,int64,object


In [4]:
genre_counts = raw["song_genre"].value_counts()
print(f"{genre_counts.size} genres, songs per genre: min {genre_counts.min()}, max {genre_counts.max()}")

114 genres, songs per genre: min 1000, max 1000


The Kaggle page mentions 125 genres, but the file contains 114, each with exactly 1,000 rows.
A perfectly even count means the data was collected genre by genre (roughly "pull 1,000 songs
tagged *X*"), not sampled from the platform as a whole. Two consequences show up below: the same
song can be pulled under several genres, and some genre lists are padded with obscure tracks.

## 2. Missing values

In [5]:
missing = raw.isna().sum()
missing[missing > 0]

artists       1
album_name    1
song_name     1
dtype: int64

In [6]:
raw[raw.isna().any(axis=1)]

,song_id,artists,album_name,song_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,song_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.460,0,0.060,0.690,0.004,0.075,0.734,138.391,4,k-pop


A single row has no artist, album or title. Its audio features are fine, but it can't be deduplicated or tied to an artist, so we drop it.

## 3. Duplicate songs

In [7]:
n_ids = raw["song_id"].nunique()
print(f"exact duplicate rows:        {raw.duplicated().sum():,}")
print(f"unique song_id:              {n_ids:,}  (out of {len(raw):,} rows)")

per_id = raw.groupby("song_id").agg(
    n_genres=("song_genre", "nunique"),
    pop_min=("popularity", "min"),
    pop_max=("popularity", "max"),
)
multi_genre = per_id[per_id["n_genres"] > 1]
pop_gap = per_id["pop_max"] - per_id["pop_min"]
print(f"song_ids listed under >1 genre: {len(multi_genre):,}")
print(f"song_ids whose copies disagree on popularity: {(pop_gap > 0).sum():,} "
      f"(median gap {pop_gap[pop_gap > 0].median():.0f} point)")

exact duplicate rows:        450
unique song_id:              89,741  (out of 114,000 rows)
song_ids listed under >1 genre: 16,299
song_ids whose copies disagree on popularity: 720 (median gap 1 point)


About 24k rows are repeats of a song already in the file, almost always because the song is listed
under more than one genre. When copies disagree on popularity, the gap is usually one point, which
looks like the scrape running over several days. We keep one row per `song_id` (the highest
popularity copy) and store the number of genres the song appeared under as `n_genres`. The full
song-to-genre mapping is saved separately in `song_genres_long.csv` in case the modeling side wants
multi-genre encoding.

Left as is, these copies would also leak across a train/test split: the model would be tested on
songs it had already seen in training.

## 4. Re-releases and the spike at popularity 0

In [8]:
print(f"share of rows with popularity 0: {(raw['popularity'] == 0).mean():.1%}")

unique_ids = raw.dropna(subset=["artists"]).drop_duplicates("song_id")
versions = unique_ids.groupby(["artists", "song_name"])["song_id"].transform("size")
zero_pop = unique_ids["popularity"] == 0
print(f"unique songs at popularity 0:                    {zero_pop.sum():,}")
print(f"  ...of which another release of the same song exists: {(zero_pop & (versions > 1)).sum():,}")

share of rows with popularity 0: 14.1%


unique songs at popularity 0:                    9,447


  ...of which another release of the same song exists: 5,455


In [9]:
example = (
    unique_ids[versions > 1]
    .groupby(["artists", "song_name"])
    .filter(lambda g: g["popularity"].max() > 60 and g["popularity"].min() == 0)
    .sort_values(["artists", "song_name", "popularity"], ascending=[True, True, False])
)
example[["artists", "song_name", "album_name", "popularity", "danceability", "energy"]].head(8)

,artists,song_name,album_name,popularity,danceability,energy
2154,3 Doors Down,Here Without You,Away From The Sun,76,0.557,0.533
2118,3 Doors Down,Here Without You,Обратно в клас - rock,0,0.556,0.545
2511,3 Doors Down,Kryptonite,The Better Life,80,0.545,0.865
44603,3 Doors Down,Kryptonite,Acoustic Back Porch Jam,52,0.608,0.835
2121,3 Doors Down,Kryptonite,Rock Workout 2023,1,0.544,0.848
2120,3 Doors Down,Kryptonite,Fishing Music,0,0.544,0.848
106014,ABBA,Dancing Queen,Arrival,83,0.543,0.870
106177,ABBA,Dancing Queen,The Essential Collection,63,0.508,0.825


14% of the raw file sits at popularity 0. Over half of those zeros are not unpopular songs. They are
duplicate releases of a song that exists elsewhere in the file with real plays: the same recording
on a single, an album and a compilation, each with its own `song_id`. Spotify attributes the plays
to one version and the rest stay near 0. *Kryptonite* by 3 Doors Down is a good example: 80 on the
original album, 52 on an acoustic compilation, and 0 on a "Fishing Music" playlist album, with
practically identical audio features.

For a model this is pure noise: identical audio features with targets of 0 and 70. We keep the
most popular release of each artist + title pair. Zeros that remain after this step are left in,
since they are songs that really get very few plays.

## 5. Values outside a sensible range

In [10]:
checks = {
    "tempo == 0 (no beat detected)": (raw["tempo"] == 0).sum(),
    "time_signature == 0": (raw["time_signature"] == 0).sum(),
    "duration < 30 s": (raw["duration_ms"] < 30_000).sum(),
    "duration > 10 min": (raw["duration_ms"] > 600_000).sum(),
    "loudness > 0 dB": (raw["loudness"] > 0).sum(),
}
pd.Series(checks, name="rows")

tempo == 0 (no beat detected)    157
time_signature == 0              163
duration < 30 s                   17
duration > 10 min                603
loudness > 0 dB                   90
Name: rows, dtype: int64

In [11]:
raw.loc[raw["tempo"] == 0, ["song_name", "song_genre", "tempo", "time_signature", "danceability"]].head()

,song_name,song_genre,tempo,time_signature,danceability
4131,The Departure,ambient,0.000,0,0.000
4379,The End of Childhood (feat. Jack Liebeck),ambient,0.000,0,0.000
4664,Ferme Les Yeux,ambient,0.000,0,0.000
45670,Campomoro,guitar,0.000,0,0.000
45720,Ritornello,guitar,0.000,0,0.000


- **tempo = 0 / time_signature = 0** (~160 rows, mostly the same songs): Spotify's audio analysis
  found no beat, and danceability is usually 0 as well. These are failed measurements, not quiet
  songs, so we drop them.
- **Under 30 seconds**: Spotify only counts a stream after 30 seconds, so popularity means
  something different for these intros and skits. Dropped (17 rows).
- **Over 10 minutes** (~600 rows): real tracks (classical, ambient, DJ mixes). We keep them. Linear
  models may want `duration_min` clipped or logged; trees won't care.
- **Loudness above 0 dB**: a handful of heavily mastered tracks. Plausible, kept.

## 6. Run the cleaning pipeline

In [12]:
clean, genres_long, cleaning_log = prepare_dataset(raw)
cleaning_log

,step,rows,rows_removed
0,raw file,114000,0
1,drop rows missing artist/album/name,113999,1
2,drop exact duplicate rows,113549,450
3,one row per song_id,89740,23809
4,one row per artist + song name,81343,8397
5,drop failed audio analysis / <30s,81181,162


In [13]:
print(f"final table: {clean.shape[0]:,} songs x {clean.shape[1]} columns")
print(f"songs at popularity 0: {(raw['popularity'] == 0).mean():.1%} before -> {(clean['popularity'] == 0).mean():.1%} after")
clean.head()

final table: 81,181 songs x 28 columns
songs at popularity 0: 14.1% before -> 5.8% after


,song_id,artists,primary_artist,album_name,song_name,song_genre,n_genres,popularity,duration_ms,duration_min,explicit,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,time_signature,n_artists,is_instrumental,is_live,speech_level,split
0,0000vdREvCVMxbQTkS888c,Rill,Rill,Lolly,Lolly,german,1,44,160725,2.679,1,0.910,0.374,-9.844,0.199,0.076,0.003,0.154,0.432,104.042,8,0,4,1,0,0,music,test
1,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,Paul Kalkbrenner,X,Böxig Leise - Pig & Dan Remix,minimal-techno,1,22,515360,8.589,0,0.686,0.560,-13.264,0.046,0.001,0.181,0.111,0.108,119.997,5,0,4,2,0,0,music,train
2,000RDCYioLteXcutOjeweY,Jordan Sandhu,Jordan Sandhu,Teeje Week,Teeje Week,hip-hop,1,62,190203,3.170,0,0.679,0.770,-3.537,0.190,0.058,0.000,0.083,0.839,161.721,0,1,4,1,0,0,music,test
3,000qpdoc97IMTBvF8gwcpy,Paul Kalkbrenner,Paul Kalkbrenner,Zeit,Tief,minimal-techno,1,19,331240,5.521,0,0.519,0.431,-13.606,0.029,0.001,0.720,0.092,0.234,129.971,6,0,4,1,1,0,music,train
4,0017XiMkqbTfF2AUOzlhj6,Chad Daniels,Chad Daniels,Busy Being Awesome,Thanksgiving Chicken,comedy,1,24,127040,2.117,1,0.536,0.780,-9.449,0.945,0.792,0.000,0.735,0.452,173.912,5,0,3,1,0,0,spoken,test


## 7. Derived columns

Every derived column uses only the song's own attributes, never the popularity score, so none of
them can leak the target.

| column | definition | why |
|---|---|---|
| `duration_min` | `duration_ms / 60000` | readable unit for charts and coefficients |
| `n_artists` | number of credited artists | tests whether collaborations help |
| `primary_artist` | first credited artist | grouping key for the split, not a model input |
| `n_genres` | genres the song was listed under | cross-genre reach (see caveat in the EDA notebook) |
| `is_instrumental` | `instrumentalness > 0.5` | Spotify's threshold for "likely no vocals" |
| `is_live` | `liveness > 0.8` | Spotify's threshold for "likely live" |
| `speech_level` | music / mixed / spoken | Spotify's 0.33 and 0.66 speechiness cut-offs |

In [14]:
clean[["duration_min", "n_artists", "n_genres", "is_instrumental", "is_live"]].describe().T

,count,mean,std,min,25%,50%,75%,max
duration_min,"81,181.000",3.859,1.920,0.501,2.900,3.588,4.457,87.288
n_artists,"81,181.000",1.374,0.881,1.000,1.000,1.000,1.000,38.000
n_genres,"81,181.000",1.223,0.588,1.000,1.000,1.000,1.000,9.000
is_instrumental,"81,181.000",0.197,0.398,0.000,0.000,0.000,0.000,1.000
is_live,"81,181.000",0.034,0.180,0.000,0.000,0.000,0.000,1.000


In [15]:
clean["speech_level"].value_counts()

speech_level
music     78131
mixed      2190
spoken      860
Name: count, dtype: int64

## 8. Train/test split

The split is fixed here so everyone in the group evaluates on the same songs. It is grouped by
`primary_artist`: all songs by an artist land on the same side. With a plain random split, a
model can learn "songs by this artist score ~70" and look accurate without learning anything about
how a song sounds, which isn't the question we are asked.

In [16]:
train, test = clean[clean["split"] == "train"], clean[clean["split"] == "test"]
overlap = set(train["primary_artist"]) & set(test["primary_artist"])
print(f"train {len(train):,} / test {len(test):,} ({len(test) / len(clean):.1%} test)")
print(f"artists on both sides: {len(overlap)}")
clean.groupby("split")["popularity"].describe()

train 65,040 / test 16,141 (19.9% test)
artists on both sides: 0


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,"16,141.000",36.351,19.805,0.000,21.000,37.000,51.000,96.000
train,"65,040.000",34.979,19.312,0.000,21.000,35.000,49.000,100.000


The two sides have similar popularity distributions, so the split doesn't favour either side.

## 9. Save

In [17]:
save_outputs(clean, genres_long)
print("saved data/processed/spotify_clean.csv and data/processed/song_genres_long.csv")

saved data/processed/spotify_clean.csv and data/processed/song_genres_long.csv
